In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
import os

In [7]:
def get_hst_area(data_in, ra_0, dec_0, ndraw, seed, drawfile, drcfits, dojwst=False):
    """
    Calculate the effective area of an HST imaging dataset using Monte Carlo sampling.
    
    Parameters
    ----------
    data_in : str
        Path to input data file containing RA, DEC columns.
    ra_0 : float
        Reference RA in degrees.
    dec_0 : float
        Reference DEC in degrees.
    ndraw : int
        Number of random draws for Monte Carlo area estimation.
    seed : int
        Random seed for reproducibility.
    drawfile : str
        Output file path for writing the valid random draws.
    badwghtfits : str
        Path to the weight/bad pixel FITS image.
    dojwst : bool, optional
        If True, read extension 1 of the FITS file; otherwise read extension 2.
        Default is False.
    
    Returns
    -------
    hstarea : float
        Estimated area in arcmin^2.
    """
    # Read input data file (RA, DEC columns, skipping comment lines starting with '#')
    data = np.loadtxt(data_in, comments='#', usecols=(0, 1))
    ra = data[:, 0]
    dec = data[:, 1]

    # Convert RA, DEC to arcseconds from RA_0, DEC_0
    xpos = 3600.0 * (ra - ra_0) * np.cos(np.radians(dec_0))
    ypos = 3600.0 * (dec - dec_0)

    # Shift so minimum is at 0
    xpos0 = -1.0 * np.min(xpos)
    ypos0 = -1.0 * np.min(ypos)
    xpos = xpos - np.min(xpos)
    ypos = ypos - np.min(ypos)
    
    # Find bounding box
    xmin = np.min(xpos)
    xmax = np.max(xpos)
    ymin = np.min(ypos)
    ymax = np.max(ypos)
    
    # Area of bounding box (in arcsec^2)
    boundarea = (xmax - xmin) * (ymax - ymin)

    # Draw random positions in the bounding box
    ndraw = int(ndraw)
    rng = np.random.default_rng(seed)
    draw_x = xmin + (xmax - xmin) * rng.uniform(size=ndraw)
    draw_y = ymin + (ymax - ymin) * rng.uniform(size=ndraw)
    
    # Convert random draws back to RA, DEC
    xdiff = draw_x - xpos0
    ydiff = draw_y - ypos0
    ra_draw = xdiff / 3600.0 / np.cos(np.radians(dec_0)) + ra_0
    dec_draw = ydiff / 3600.0 + dec_0
    
    # Read the weight/bad pixel FITS image
    if dojwst:
        exten = 1
    else:
        exten = 2
    
    hdu_list = fits.open(drcfits)
    badwght = hdu_list[exten].data.astype(float)
    badhdr = hdu_list[exten].header
    hdu_list.close()

    # Replace NaN and Inf values with 0
    badwght[~np.isfinite(badwght)] = 0.0
    
    # Convert RA, DEC of random draws to pixel coordinates using WCS
    wcs = WCS(badhdr)
    # adxy equivalent: world to pixel
    ximg_draw, yimg_draw = wcs.all_world2pix(ra_draw, dec_draw, 0)
    
    # Convert to integer pixel indices
    ximg_draw = ximg_draw.astype(int)
    yimg_draw = yimg_draw.astype(int)
    
    # Ensure pixel indices are within image bounds
    ny, nx = badwght.shape
    in_bounds = (
        (ximg_draw >= 0) & (ximg_draw < nx) &
        (yimg_draw >= 0) & (yimg_draw < ny)
    )

    # Check which draws land on valid (non-zero weight) pixels
    # Note: FITS data in Python is accessed as [y, x]
    #gdindex = np.where(
    #    in_bounds & (badwght[yimg_draw[in_bounds] if in_bounds.all() else np.clip(yimg_draw, 0, ny-1),
    #                         ximg_draw[in_bounds] if in_bounds.all() else np.clip(ximg_draw, 0, nx-1)] != 0.0)
    #)[0]
    
    # Simpler, safer approach:
    # Filter to in-bounds first, then check pixel values
    valid_mask = np.zeros(ndraw, dtype=bool)
    for i in range(ndraw):
        if in_bounds[i]:
            if badwght[yimg_draw[i], ximg_draw[i]] != 0.0:
                valid_mask[i] = True
    
    count_gddraw = np.sum(valid_mask)
    
    print(f"Fraction of valid draws: {count_gddraw / ndraw}")
    print(f"Bounding area (arcsec^2): {boundarea}")
    
    # Calculate effective area

    area = boundarea * (count_gddraw / ndraw)
    
    print(f"HST Area is {area / 3600.0} arcmin squared")
    
    # Write output file
    write_ra = ra_draw[valid_mask]
    write_dec = dec_draw[valid_mask]
    
    if os.path.exists(drawfile):
        os.remove(drawfile)
    
    with open(drawfile, 'w') as f:
        f.write(f"# HSTAREA {area / 3600.0} arcmin squared\n")
        for i in range(len(write_ra)):
            f.write(f"{write_ra[i]} {write_dec[i]}\n")
    
    return area / 3600.0

In [8]:
area_arcsec = get_hst_area('/Users/dsand/Dropbox/NGC300Dw_HST/StructuralParametersSculptorDW/NGC300DW2/NGC300DW2_RGBmap.dat',
                               16.712083,-35.077500,1000000,12,'good_draw_NGC300DW2_test.txt',
                           '/Users/dsand/Dropbox/NGC300Dw_HST/StructuralParametersSculptorDW/NGC300DW2/NGC300DW2_F814W_drc.fits')

Fraction of valid draws: 0.71804
Bounding area (arcsec^2): 53153.42535313488
HST Area is 10.601745983490268 arcmin squared
